In [1]:
import pandas as pd
import numpy as np
import os
from scipy.signal import savgol_filter

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# -----------------------------
# Load data
# -----------------------------
train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# -----------------------------
# Define spectral columns
# -----------------------------
spectral_cols = [c for c in train.columns if c not in ['sample number', 'species number', '樹種', '含水率']]
print("Spectral features:", len(spectral_cols))

# -----------------------------
# Extract X, y, groups
# -----------------------------
X = train[spectral_cols].values
y = train['含水率'].values
groups = train['species number'].values
X_test = test[spectral_cols].values

# -----------------------------
# Apply Savitzky-Golay first derivative
# -----------------------------
X_sg = savgol_filter(X, window_length=11, polyorder=2, deriv=1)
X_test_sg = savgol_filter(X_test, window_length=11, polyorder=2, deriv=1)

# -----------------------------
# Feature scaling
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sg)
X_test_scaled = scaler.transform(X_test_sg)
print("Feature scaling complete")

# -----------------------------
# Group-aware cross-validation
# -----------------------------
kf = GroupKFold(n_splits=5)
rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_scaled, y, groups=groups), 1):
    print(f"\nTraining fold {fold}")
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = ElasticNet(alpha=1.0, l1_ratio=0.7, max_iter=100000, tol=1e-3, random_state=42)
    model.fit(X_train, y_train)

    val_preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    print("Fold RMSE:", rmse)
    rmse_scores.append(rmse)

print("\nMean GroupKFold CV RMSE:", np.mean(rmse_scores))

# -----------------------------
# Train final model on full data
# -----------------------------
print("\nTraining final model on full dataset...")
final_model = ElasticNet(alpha=1.0, l1_ratio=0.7, max_iter=100000, tol=1e-3, random_state=42)
final_model.fit(X_scaled, y)

# -----------------------------
# Predict test set
# -----------------------------
test_preds = final_model.predict(X_test_scaled)
print("Sample predictions:", test_preds[:10])

# -----------------------------
# Create submission
# -----------------------------
submission = pd.DataFrame({
    "sample number": test["sample number"],
    "含水率": test_preds
})

# -----------------------------
# Save submission
# -----------------------------
os.makedirs("../submissions", exist_ok=True)
output_file = "../submissions/exp14_elasticnet_savgol_groupkfold_20260323.csv"
submission.to_csv(output_file, index=False, header=False)
print("Submission saved to:", output_file)

# -----------------------------
# Verify saved file
# -----------------------------
check = pd.read_csv(output_file, header=None)
print(check.head())

Train shape: (1322, 1559)
Test shape: (550, 1558)
Spectral features: 1555
Feature scaling complete

Training fold 1
Fold RMSE: 28.031211754671073

Training fold 2
Fold RMSE: 29.171752638547872

Training fold 3
Fold RMSE: 47.97615618412902

Training fold 4
Fold RMSE: 54.3419577587072

Training fold 5
Fold RMSE: 15.927062169178281

Mean GroupKFold CV RMSE: 35.089628101046685

Training final model on full dataset...
Sample predictions: [203.54620999 166.21511326 155.52318903 146.96093994 141.39895305
 134.98378829 125.30656503 127.18494403 123.35666303 119.12847951]
Submission saved to: ../submissions/exp14_elasticnet_savgol_groupkfold_20260323.csv
    0           1
0  95  203.546210
1  96  166.215113
2  97  155.523189
3  98  146.960940
4  99  141.398953
